# 00 — Préparer les données et les splits

Ce notebook construit le **contrat de données unique** utilisé par tous les modèles image.

Objectifs :

- joindre les caractéristiques et les labels par `Id` ;
- vérifier l'existence et la lisibilité de chaque image ;
- détecter les images uniformes et les doublons perceptuels ;
- empêcher un même groupe de doublons d'apparaître dans train et validation ;
- exclure les groupes impossibles à apprendre, associés à plusieurs labels ;
- sauvegarder un manifeste reproductible et le mapping des 27 classes.

Le calcul du pHash sur toutes les images est coûteux la première fois. Son résultat est mis en cache.


In [8]:
from pathlib import Path
import json
import os
import random

import certifi
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Python installé depuis python.org sur macOS ne référence pas toujours le trousseau
# système. On fournit à urllib/Keras un bundle CA explicite sans désactiver SSL.
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

def find_project_root() -> Path:
    """Permet d'exécuter le notebook depuis sa racine ou son propre dossier."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "raw_x.csv").exists() and (candidate / "images").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable (data/raw_x.csv et images/ attendus).")

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
IMAGE_DIR = ROOT / "images" / "image_train"
ARTIFACTS_DIR = ROOT / "artifacts" / "image_classification"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("KERAS_HOME", str(ARTIFACTS_DIR / ".keras"))

print("Racine :", ROOT)
print("Artefacts :", ARTIFACTS_DIR)
print("Certificats SSL :", os.environ["SSL_CERT_FILE"])


Racine : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2
Artefacts : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification
Certificats SSL : /Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/certifi/cacert.pem


## 1. Charger et joindre les tables

Une jointure explicite sur `Id` évite le risque silencieux d'un désalignement entre X et Y.


In [9]:
raw_x = pd.read_csv(DATA_DIR / "raw_x.csv")
raw_y = pd.read_csv(DATA_DIR / "raw_y.csv")

required_x = {"Id", "productid", "imageid"}
required_y = {"Id", "prdtypecode"}
assert required_x.issubset(raw_x.columns), required_x - set(raw_x.columns)
assert required_y.issubset(raw_y.columns), required_y - set(raw_y.columns)
assert raw_x["Id"].is_unique and raw_y["Id"].is_unique

data = raw_x.merge(raw_y, on="Id", how="inner", validate="one_to_one")
assert len(data) == len(raw_x) == len(raw_y)
data["image_filename"] = (
    "image_" + data["imageid"].astype(str)
    + "_product_" + data["productid"].astype(str) + ".jpg"
)
data["image_path"] = data["image_filename"].map(lambda name: str((IMAGE_DIR / name).resolve()))
data.head()


,Id,designation,description,productid,imageid,prdtypecode,image_filename,image_path
0,0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,10,image_1263597046_product_3804725264.jpg,/Users/wolfidy7/projects/sep25_alt2_iia_rakute...
1,1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,2280,image_1008141237_product_436067568.jpg,/Users/wolfidy7/projects/sep25_alt2_iia_rakute...
2,2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,50,image_938777978_product_201115110.jpg,/Users/wolfidy7/projects/sep25_alt2_iia_rakute...
3,3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,1280,image_457047496_product_50418756.jpg,/Users/wolfidy7/projects/sep25_alt2_iia_rakute...
4,4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,2705,image_1077757786_product_278535884.jpg,/Users/wolfidy7/projects/sep25_alt2_iia_rakute...


In [10]:
exists = data["image_path"].map(os.path.isfile)
print(f"Images trouvées : {exists.sum():,}/{len(data):,}")
if not exists.all():
    display(data.loc[~exists, ["Id", "image_path"]].head())
    raise FileNotFoundError(f"{(~exists).sum()} images sont absentes.")

counts = data["prdtypecode"].value_counts().sort_index()
display(counts.rename("effectif").to_frame())
print("Ratio classe majoritaire/minoritaire :", round(counts.max() / counts.min(), 2))


Images trouvées : 84,916/84,916


,effectif
prdtypecode,
10,3116
40,2508
50,1681
60,832
1140,2671
1160,3953
1180,764
1280,4870
1281,2070


Ratio classe majoritaire/minoritaire : 13.36


## 2. Qualité et empreinte perceptuelle

Le pHash identifie ici les images visuellement identiques avec la même empreinte. Les pixels ne sont
pas chargés en mémoire simultanément. Une image uniforme est définie par un écart-type en niveaux de
gris inférieur à 5, en cohérence avec l'EDA existante.


In [11]:
import hashlib
from PIL import Image, UnidentifiedImageError
import imagehash
from tqdm.auto import tqdm

HASH_CACHE = ARTIFACTS_DIR / "image_hashes.csv"

def inspect_image(path):
    try:
        with Image.open(path) as image:
            image = image.convert("RGB")
            gray = np.asarray(image.convert("L"), dtype=np.float32)
            return str(imagehash.phash(image)), float(gray.std()), True
    except (OSError, UnidentifiedImageError):
        return None, np.nan, False

if HASH_CACHE.exists():
    image_info = pd.read_csv(HASH_CACHE)
    print("Cache chargé :", HASH_CACHE)
else:
    rows = []
    for row in tqdm(data[["Id", "image_path"]].itertuples(index=False), total=len(data)):
        phash, gray_std, readable = inspect_image(row.image_path)
        rows.append({"Id": row.Id, "phash": phash, "gray_std": gray_std, "readable": readable})
    image_info = pd.DataFrame(rows)
    image_info.to_csv(HASH_CACHE, index=False)

assert image_info["Id"].is_unique
data = data.merge(image_info, on="Id", validate="one_to_one")
data["uniform_image"] = data["gray_std"] < 5
print(data[["readable", "uniform_image"]].value_counts())


Cache chargé : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification/image_hashes.csv
readable  uniform_image
True      False            84866
          True                50
Name: count, dtype: int64


## 3. Construire les groupes et contrôler les conflits

Toutes les lignes partageant le même pHash reçoivent le même `duplicate_group`. Si une même image est
associée à plusieurs codes produit, le signal visuel est contradictoire : le groupe est conservé dans
le rapport d'audit, mais exclu de l'entraînement supervisé image.


In [12]:
data["duplicate_group"] = pd.factorize(data["phash"], sort=True)[0]
group_stats = data.groupby("duplicate_group").agg(
    group_size=("Id", "size"),
    label_count=("prdtypecode", "nunique"),
)
data = data.join(group_stats, on="duplicate_group")
data["label_conflict"] = data["label_count"] > 1
data["eligible"] = data["readable"] & ~data["uniform_image"] & ~data["label_conflict"]

audit = pd.Series({
    "lignes_total": len(data),
    "images_illisibles": (~data["readable"]).sum(),
    "images_uniformes": data["uniform_image"].sum(),
    "lignes_dans_groupes_dupliques": (data["group_size"] > 1).sum(),
    "groupes_avec_labels_contradictoires": group_stats["label_count"].gt(1).sum(),
    "lignes_eligibles": data["eligible"].sum(),
})
display(audit.to_frame("valeur"))


,valeur
lignes_total,84916
images_illisibles,0
images_uniformes,50
lignes_dans_groupes_dupliques,11517
groupes_avec_labels_contradictoires,793
lignes_eligibles,82375


## 4. Split stratifié et groupé

`StratifiedGroupKFold` préserve au mieux les proportions de classes tout en garantissant qu'un groupe
de doublons ne traverse jamais la frontière train/validation. Le fold 0 est utilisé comme validation.

Le `sample_weight` inverse la taille du groupe : dix copies de la même image ne pèseront pas dix fois
plus qu'une image unique. Les poids sont renormalisés autour de 1 pour garder une échelle de loss stable.


In [13]:
from sklearn.model_selection import StratifiedGroupKFold

model_data = data.loc[data["eligible"]].copy().reset_index(drop=True)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, val_idx = next(splitter.split(
    model_data,
    y=model_data["prdtypecode"],
    groups=model_data["duplicate_group"],
))
model_data["split"] = "train"
model_data.loc[val_idx, "split"] = "validation"

overlap = set(model_data.loc[train_idx, "duplicate_group"]) & set(model_data.loc[val_idx, "duplicate_group"])
assert not overlap
assert set(model_data["prdtypecode"].unique()) == set(model_data.loc[val_idx, "prdtypecode"].unique())

model_data["sample_weight"] = 1.0 / model_data["group_size"]
train_mean = model_data.loc[model_data["split"] == "train", "sample_weight"].mean()
model_data["sample_weight"] /= train_mean

split_table = pd.crosstab(model_data["prdtypecode"], model_data["split"])
split_table["validation_pct"] = 100 * split_table["validation"] / split_table.sum(axis=1)
display(split_table)


split,train,validation,validation_pct
prdtypecode,,,
10,2364,591,20.000000
40,1902,476,20.016821
50,1280,320,20.000000
60,659,164,19.927096
1140,2037,510,20.023557
1160,3072,768,20.000000
1180,593,148,19.973009
1280,3586,897,20.008923
1281,1588,396,19.959677


## 5. Sauvegarder le contrat de données

Les codes métier ne sont pas nécessairement contigus. Le mapping les transforme en indices `0..26`
pour la couche softmax et permet ensuite de revenir au `prdtypecode` original.


In [14]:
columns = [
    "Id", "productid", "imageid", "image_path", "prdtypecode", "phash",
    "duplicate_group", "group_size", "sample_weight", "split",
]
manifest = model_data[columns].copy()
manifest.to_csv(ARTIFACTS_DIR / "image_manifest.csv", index=False)

index_to_code = sorted(int(code) for code in manifest["prdtypecode"].unique())
mapping = {"index_to_code": index_to_code}
with open(ARTIFACTS_DIR / "class_mapping.json", "w", encoding="utf-8") as handle:
    json.dump(mapping, handle, indent=2, ensure_ascii=False)

split_table.to_csv(ARTIFACTS_DIR / "split_summary.csv")
data.to_csv(ARTIFACTS_DIR / "image_quality_audit.csv", index=False)
print("Manifeste :", ARTIFACTS_DIR / "image_manifest.csv")
print("Mapping   :", ARTIFACTS_DIR / "class_mapping.json")


Manifeste : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification/image_manifest.csv
Mapping   : /Users/wolfidy7/projects/sep25_alt2_iia_rakuten_2/artifacts/image_classification/class_mapping.json
